# Standalone lineage graph from exported VTP

This notebook loads one `.vtp` mesh exported by `scripts/export_marker_intensities_to_mesh.py`, reconstructs the cell adjacency graph from the mesh surface ownership field, and plots cells colored by lineage.

It does not import `organograph` or any project-local code. The only required external packages are `numpy`, `pandas`, `networkx`, `plotly`, and `vtk`.

If needed, install the runtime dependencies with:

```bash
pip install numpy pandas networkx plotly vtk
```


In [ ]:
from pathlib import Path

# Set this to the exported VTP file you want to inspect.
VTP_PATH = Path("path/to/exported_mesh_with_lineages.vtp")

# Field written by export_marker_intensities_to_mesh.py. Each mesh vertex stores
# the original cell-table index of the cell whose surface patch owns that vertex.
OWNER_FIELD = "cell_owner_table_index"
UNASSIGNED_OWNER = -1

# Match the OrganoGraph preprocessing rule: skip triangles where any vertex is unowned.
REQUIRE_ALL_TRIANGLE_VERTICES_OWNED = True

# If None, lineage fields are auto-detected. For collaborator-exclusivity exports
# this normally resolves to STEM_intensity, EEPROG_intensity, ..., TA_intensity.
LINEAGE_FIELDS = None
LINEAGE_PRIORITY = [
    "STEM",
    "EEPROG",
    "GOBLET",
    "ABS",
    "EE",
    "SECPROG",
    "EC",
    "PANETH",
    "TA",
]

LINEAGE_COLORS = {
    "STEM": "#1b9e77",
    "EEPROG": "#7570b3",
    "GOBLET": "#e7298a",
    "ABS": "#66a61e",
    "EE": "#e6ab02",
    "SECPROG": "#a6761d",
    "EC": "#d95f02",
    "PANETH": "#e41a1c",
    "TA": "#377eb8",
}
BASELINE_LABEL = "unassigned"
BASELINE_COLOR = "#bdbdbd"
UNOWNED_SURFACE_COLOR = "#eeeeee"

GRAPH_NODE_SIZE = 4
GRAPH_EDGE_WIDTH = 0.8
MESH_OPACITY = 0.95
FIG_WIDTH = 900
FIG_HEIGHT = 720


In [ ]:
import itertools
import re

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go

try:
    import vtk
    from vtk.util.numpy_support import vtk_to_numpy
except ImportError as exc:
    raise ImportError(
        "This standalone notebook needs the 'vtk' Python package to read .vtp files. "
        "Install it with: pip install vtk"
    ) from exc


## Standalone helper functions

In [ ]:
def read_vtp_mesh(path):
    """Read a VTP PolyData mesh and return vertices, triangular faces, and point fields."""
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"VTP file not found: {path}")

    reader = vtk.vtkXMLPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    polydata = reader.GetOutput()
    if polydata is None or polydata.GetNumberOfPoints() == 0:
        raise ValueError(f"No points found in VTP file: {path}")

    vertices = vtk_to_numpy(polydata.GetPoints().GetData()).astype(float, copy=False)

    faces = []
    polys = polydata.GetPolys()
    polys.InitTraversal()
    id_list = vtk.vtkIdList()
    while polys.GetNextCell(id_list):
        ids = [int(id_list.GetId(i)) for i in range(id_list.GetNumberOfIds())]
        if len(ids) < 3:
            continue
        if len(ids) == 3:
            faces.append(ids)
        else:
            # Fan-triangulate polygons just in case a non-triangular cell appears.
            for j in range(1, len(ids) - 1):
                faces.append([ids[0], ids[j], ids[j + 1]])
    faces = np.asarray(faces, dtype=np.int64)

    point_data = polydata.GetPointData()
    point_fields = {}
    for i in range(point_data.GetNumberOfArrays()):
        arr = point_data.GetArray(i)
        if arr is None:
            continue
        name = arr.GetName() or f"field_{i}"
        values = vtk_to_numpy(arr)
        if values.ndim == 2 and values.shape[1] == 1:
            values = values[:, 0]
        point_fields[name] = np.asarray(values)

    return vertices, faces, point_fields


def sanitize_field_name(name):
    """Match the field-name sanitization used by export_marker_intensities_to_mesh.py."""
    out = re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")
    out = re.sub(r"_+", "_", out)
    if not out:
        out = "marker"
    if out[0].isdigit():
        out = f"marker_{out}"
    return out


def infer_lineage_fields(point_fields, lineage_fields=None, priority=None):
    """Resolve lineage point-data fields from the VTP point-data arrays."""
    field_names = list(point_fields)
    if lineage_fields is not None:
        missing = [name for name in lineage_fields if name not in point_fields]
        if missing:
            raise KeyError(f"Requested lineage fields missing from VTP: {missing}")
        lineage_names = [field_to_lineage_name(name) for name in lineage_fields]
        return list(lineage_fields), lineage_names

    priority = list(priority or [])
    canonical = []
    for lineage in priority:
        field = f"{sanitize_field_name(lineage)}_intensity"
        if field in point_fields:
            canonical.append(field)
    if canonical:
        return canonical, [field_to_lineage_name(name) for name in canonical]

    detected = [
        name for name in field_names
        if name.endswith("_intensity") and name != OWNER_FIELD
    ]
    if not detected:
        raise ValueError(
            "Could not auto-detect lineage fields. Expected fields ending in '_intensity'. "
            f"Available point fields are: {field_names}"
        )
    return detected, [field_to_lineage_name(name) for name in detected]


def field_to_lineage_name(field):
    name = str(field)
    if name.endswith("_intensity"):
        name = name[:-len("_intensity")]
    if name.startswith("marker_"):
        name = name[len("marker_"):]
    return name


def aggregate_vertex_fields_by_owner(owners, vertex_values, owner_ids):
    """Aggregate per-vertex lineage intensities into per-cell values by max over owned vertices."""
    owners = np.asarray(owners, dtype=np.int64)
    vertex_values = np.asarray(vertex_values, dtype=float)
    owner_ids = np.asarray(owner_ids, dtype=np.int64)
    out = np.zeros((owner_ids.size, vertex_values.shape[1]), dtype=float)
    for row, owner in enumerate(owner_ids):
        mask = owners == int(owner)
        if np.any(mask):
            out[row] = np.nanmax(vertex_values[mask], axis=0)
    return out


def assign_lineage_labels(lineage_intensities, lineage_names, priority=None, baseline_label=BASELINE_LABEL):
    """Assign one display label per cell from positive lineage intensities."""
    X = np.asarray(lineage_intensities, dtype=float)
    lineage_names = list(lineage_names)
    priority = list(priority or lineage_names)
    priority_rank = {name: i for i, name in enumerate(priority)}
    labels = np.full(X.shape[0], baseline_label, dtype=object)

    for i in range(X.shape[0]):
        positive_idx = np.flatnonzero(X[i] > 0)
        if positive_idx.size == 0:
            continue
        if positive_idx.size == 1:
            labels[i] = lineage_names[int(positive_idx[0])]
            continue
        max_value = np.nanmax(X[i, positive_idx])
        tied = [int(j) for j in positive_idx if X[i, j] == max_value]
        tied.sort(key=lambda j: priority_rank.get(lineage_names[j], len(priority_rank) + j))
        labels[i] = lineage_names[tied[0]]
    return labels


def build_lineage_graph_from_owner_mesh(
    vertices,
    faces,
    owners,
    lineage_vertex_values,
    lineage_names,
    lineage_priority=None,
    require_all_triangle_vertices_owned=True,
):
    """
    Build a cell adjacency graph from mesh vertex ownership.

    Nodes are cell owners from the VTP owner field. Edges connect two cells when
    their owned surface patches touch on a mesh triangle. This mirrors the graph
    construction rule used in the OrganoGraph preprocessing code.
    """
    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=np.int64)
    owners = np.asarray(owners, dtype=np.int64).reshape(-1)
    if owners.shape[0] != vertices.shape[0]:
        raise ValueError(f"owners has length {owners.shape[0]} but mesh has {vertices.shape[0]} vertices")

    owner_ids = np.array(sorted(int(x) for x in np.unique(owners) if int(x) >= 0), dtype=np.int64)
    owner_to_node = {int(owner): int(i) for i, owner in enumerate(owner_ids)}
    node_to_owner = {int(i): int(owner) for i, owner in enumerate(owner_ids)}

    lineage_intensities = aggregate_vertex_fields_by_owner(owners, lineage_vertex_values, owner_ids)
    labels = assign_lineage_labels(
        lineage_intensities,
        lineage_names,
        priority=lineage_priority,
        baseline_label=BASELINE_LABEL,
    )

    G = nx.Graph()
    for node, owner in node_to_owner.items():
        owned_vertices = np.flatnonzero(owners == owner)
        centroid = vertices[owned_vertices].mean(axis=0) if owned_vertices.size else np.full(3, np.nan)
        G.add_node(
            node,
            cell_index=int(owner),
            owner_id=int(owner),
            centroid=centroid,
            n_owned_vertices=int(owned_vertices.size),
            lineage_label=str(labels[node]),
            lineage_intensities=lineage_intensities[node].astype(float),
        )

    for tri in faces:
        tri_owners = owners[tri]
        valid = tri_owners >= 0
        if require_all_triangle_vertices_owned and not np.all(valid):
            continue
        tri_owners = tri_owners[valid]
        tri_nodes = [owner_to_node[int(owner)] for owner in tri_owners if int(owner) in owner_to_node]
        for a, b in itertools.combinations(tri_nodes, 2):
            if a != b:
                G.add_edge(int(a), int(b))

    aux = {
        "owner_ids": owner_ids,
        "owner_to_node": owner_to_node,
        "node_to_owner": node_to_owner,
        "lineage_intensities": lineage_intensities,
        "lineage_labels": labels,
    }
    G.graph["lineage_names"] = list(lineage_names)
    return G, aux


In [ ]:
def lineage_color(label):
    return LINEAGE_COLORS.get(str(label), BASELINE_COLOR if label == BASELINE_LABEL else "#333333")


def set_equal_3d_layout(fig, points, width=FIG_WIDTH, height=FIG_HEIGHT):
    points = np.asarray(points, dtype=float)
    center = np.nanmean(points, axis=0)
    span = np.nanmax(points, axis=0) - np.nanmin(points, axis=0)
    radius = float(np.nanmax(span) / 2) if np.all(np.isfinite(span)) else 1.0
    radius = max(radius, 1e-9)
    ranges = [[center[i] - radius, center[i] + radius] for i in range(3)]
    fig.update_layout(
        width=width,
        height=height,
        scene=dict(
            xaxis=dict(visible=False, range=ranges[0]),
            yaxis=dict(visible=False, range=ranges[1]),
            zaxis=dict(visible=False, range=ranges[2]),
            aspectmode="cube",
        ),
        margin=dict(l=0, r=0, t=45, b=0),
    )
    return fig


def graph_lineage_summary(G):
    rows = []
    for label, nodes in sorted(
        ((label, [n for n, d in G.nodes(data=True) if d.get("lineage_label") == label])
         for label in {d.get("lineage_label") for _, d in G.nodes(data=True)}),
        key=lambda item: (item[0] == BASELINE_LABEL, str(item[0])),
    ):
        rows.append({
            "lineage": label,
            "n_cells": len(nodes),
            "fraction": len(nodes) / max(G.number_of_nodes(), 1),
        })
    return pd.DataFrame(rows)


def plot_lineage_graph(G, title="Cell adjacency graph colored by lineage"):
    positions = np.vstack([np.asarray(G.nodes[n]["centroid"], dtype=float) for n in G.nodes])
    node_index = {node: i for i, node in enumerate(G.nodes)}

    edge_x, edge_y, edge_z = [], [], []
    for a, b in G.edges:
        pa = positions[node_index[a]]
        pb = positions[node_index[b]]
        edge_x += [pa[0], pb[0], None]
        edge_y += [pa[1], pb[1], None]
        edge_z += [pa[2], pb[2], None]

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode="lines",
        line=dict(color="rgba(80,80,80,0.35)", width=GRAPH_EDGE_WIDTH),
        hoverinfo="skip",
        name="adjacency",
        showlegend=False,
    ))

    labels = np.array([G.nodes[n].get("lineage_label", BASELINE_LABEL) for n in G.nodes], dtype=object)
    ordered_labels = [name for name in LINEAGE_PRIORITY if name in set(labels)]
    ordered_labels += [label for label in sorted(set(labels)) if label not in ordered_labels and label != BASELINE_LABEL]
    if BASELINE_LABEL in set(labels):
        ordered_labels.append(BASELINE_LABEL)

    for label in ordered_labels:
        node_mask = labels == label
        node_ids = np.array(list(G.nodes), dtype=object)[node_mask]
        pts = positions[node_mask]
        hover = [
            f"cell_index: {G.nodes[n]['cell_index']}<br>lineage: {label}<br>degree: {G.degree[n]}<br>owned vertices: {G.nodes[n]['n_owned_vertices']}"
            for n in node_ids
        ]
        fig.add_trace(go.Scatter3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            mode="markers",
            marker=dict(size=GRAPH_NODE_SIZE, color=lineage_color(label), line=dict(width=0)),
            text=hover,
            hoverinfo="text",
            name=str(label),
        ))

    fig.update_layout(title=title, legend_title_text="Lineage")
    return set_equal_3d_layout(fig, positions)


def plot_lineage_mesh(vertices, faces, owners, G, aux, title="Mesh surface patches colored by lineage"):
    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=np.int64)
    owners = np.asarray(owners, dtype=np.int64)
    owner_to_node = aux["owner_to_node"]

    face_labels = []
    for tri in faces:
        tri_owners = owners[tri]
        tri_owners = tri_owners[tri_owners >= 0]
        if tri_owners.size == 0:
            face_labels.append(None)
            continue
        unique, counts = np.unique(tri_owners, return_counts=True)
        owner = int(unique[np.argmax(counts)])
        node = owner_to_node.get(owner, None)
        face_labels.append(G.nodes[node]["lineage_label"] if node is not None else None)

    face_colors = [lineage_color(label) if label is not None else UNOWNED_SURFACE_COLOR for label in face_labels]
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        facecolor=face_colors,
        opacity=MESH_OPACITY,
        flatshading=True,
        hoverinfo="skip",
        name="cell patches",
        showlegend=False,
    ))

    present_labels = [name for name in LINEAGE_PRIORITY if name in set(face_labels)]
    present_labels += [label for label in sorted({x for x in face_labels if x is not None}) if label not in present_labels and label != BASELINE_LABEL]
    if BASELINE_LABEL in set(face_labels):
        present_labels.append(BASELINE_LABEL)
    for label in present_labels:
        fig.add_trace(go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode="markers",
            marker=dict(size=8, color=lineage_color(label)),
            name=str(label),
            showlegend=True,
        ))

    fig.update_layout(title=title, legend_title_text="Lineage")
    return set_equal_3d_layout(fig, vertices)


## Load the VTP and build the graph

In [ ]:
vertices, faces, point_fields = read_vtp_mesh(VTP_PATH)

if OWNER_FIELD not in point_fields:
    raise KeyError(
        f"Owner field {OWNER_FIELD!r} not found. Available point fields: {list(point_fields)}"
    )
owners = np.asarray(point_fields[OWNER_FIELD], dtype=np.int64).reshape(-1)

lineage_fields, lineage_names = infer_lineage_fields(
    point_fields,
    lineage_fields=LINEAGE_FIELDS,
    priority=LINEAGE_PRIORITY,
)
lineage_vertex_values = np.column_stack([
    np.asarray(point_fields[field], dtype=float).reshape(-1)
    for field in lineage_fields
])

G, graph_aux = build_lineage_graph_from_owner_mesh(
    vertices,
    faces,
    owners,
    lineage_vertex_values,
    lineage_names,
    lineage_priority=LINEAGE_PRIORITY,
    require_all_triangle_vertices_owned=REQUIRE_ALL_TRIANGLE_VERTICES_OWNED,
)

print(f"Loaded: {VTP_PATH}")
print(f"Mesh vertices: {vertices.shape[0]:,}")
print(f"Mesh faces: {faces.shape[0]:,}")
print(f"Lineage fields: {lineage_fields}")
print(f"Graph nodes/cells: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")
display(graph_lineage_summary(G))


## Plot the lineage graph

In [ ]:
fig_graph = plot_lineage_graph(G, title=f"{VTP_PATH.name} - lineage graph")
fig_graph


## Plot the mesh surface patches

In [ ]:
fig_mesh = plot_lineage_mesh(vertices, faces, owners, G, graph_aux, title=f"{VTP_PATH.name} - lineage surface patches")
fig_mesh


## Optional: inspect the graph object

In [ ]:
# The graph is a standard networkx.Graph. Node ids are contiguous integers, and
# each node stores the original owner/cell-table index as node["cell_index"].
example_nodes = list(G.nodes)[:5]
for node in example_nodes:
    print(node, G.nodes[node])
